In [13]:
# TinySQL playground: ask questions about a real dataset (Palmer penguins, 344 rows)
import re, sqlite3
import pandas as pd, regex, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

REPO = "adiitya0929/TinySQL-125M-WikiSQL"
tokenizer = AutoTokenizer.from_pretrained(REPO)
model = AutoModelForCausalLM.from_pretrained(REPO)
END = tokenizer.convert_tokens_to_ids("</sql>")

# 1. Constrained decoding: only real column names, and values copied from the question
class SqlGrammar(LogitsProcessor):
    def __init__(self, columns, prompt_len, question=None):
        col = "(?:" + "|".join(regex.escape('"' + c.replace('"', '""') + '"') for c in columns) + ")"
        sel = rf"(?:(?:MAX|MIN|COUNT|SUM|AVG)\({col}\)|{col})"
        cond = rf"{col} (?:=|>|<) '(?:[^']|'')*'"
        self.sql = regex.compile(rf"SELECT {sel} FROM t(?: WHERE {cond}(?: AND {cond})*)?;")
        self.prompt_len, self.q = prompt_len, None if question is None else question.lower()

    def ok(self, text):
        text = text.rstrip("\ufffd")                          # half of a multi-byte character: judge the rest
        if not self.sql.fullmatch(text, partial=True):        # could this still become a valid query?
            return False
        if self.q is not None:
            for v in regex.findall(r"(?:=|>|<) '((?:[^']|'')*)", text):
                if v.replace("''", "'").lower() not in self.q:
                    return False
        return True

    def __call__(self, input_ids, scores):
        for b in range(input_ids.shape[0]):
            gen = input_ids[b, self.prompt_len:].tolist()
            keep = END if self.sql.fullmatch(tokenizer.decode(gen)) else None
            if keep is None:
                for t in torch.argsort(scores[b], descending=True).tolist():
                    if t not in tokenizer.all_special_ids and self.ok(tokenizer.decode(gen + [t])):
                        keep = t
                        break
            mask = torch.full_like(scores[b], float("-inf"))
            mask[keep if keep is not None else END] = 0
            scores[b] = scores[b] + mask
        return scores

def text_to_sql(df, question, constrained=True):
    cols = ", ".join(f'"{c}" {"REAL" if pd.api.types.is_numeric_dtype(df[c]) else "TEXT"}' for c in df.columns)
    prompt = f"<schema>CREATE TABLE t ({cols});</schema>\n<question>{question}</question>\n<sql>"
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    procs = LogitsProcessorList([SqlGrammar(list(df.columns), ids.shape[1], question)]) if constrained else None
    out = model.generate(ids, max_new_tokens=64, do_sample=False, logits_processor=procs)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

# 2. Grounding: map each text value to a real cell value ('Gentoo penguin' -> 'Gentoo', 'Torgersen' -> Island)
COND = re.compile(r'"((?:[^"]|"")+)"\s*(=|>|<)\s*\'((?:[^\']|\'\')*)\'')

def ground(df, sql):
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    cells = {c: df[c].dropna().astype(str).unique() for c in text_cols}
    def word_in(cell, val):
        return re.search(r"(?<![a-z0-9])" + re.escape(cell.lower()) + r"(?![a-z0-9])", val.lower())
    def fix(m):
        col, op, val = m.group(1).replace('""', '"'), m.group(2), m.group(3).replace("''", "'").strip()
        if op == "=" and col in cells and not any(c.lower() == val.lower() for c in cells[col]):
            for search in [col] + [c for c in text_cols if c != col]:      # the chosen column first
                hits = [c for c in cells[search] if word_in(c, val)]
                if hits:
                    col, val = search, max(hits, key=len)
                    break
        return '"' + col.replace('"', '""') + '" ' + op + " '" + val.replace("'", "''") + "'"
    return COND.sub(fix, sql)

# 3. Run SQL on the DataFrame (table t, case-insensitive text comparison)
def run_sql(df, sql):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE t (" + ", ".join(
        f'"{c}" {"REAL" if pd.api.types.is_numeric_dtype(df[c]) else "TEXT COLLATE NOCASE"}' for c in df.columns) + ")")
    df.to_sql("t", con, if_exists="append", index=False)
    try:
        return pd.read_sql_query(sql, con).values.tolist()[:5]
    except Exception as e:
        return f"error: {e}"

# 4. Data and questions
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
df.columns = ["Species", "Island", "Bill Length (mm)", "Bill Depth (mm)",
              "Flipper Length (mm)", "Body Mass (g)", "Sex"]

questions = [
    "What is the heaviest body mass of a Gentoo penguin?",
    "What is the average flipper length of Adelie penguins?",
    "What is the lowest body mass on Torgersen island?",
    "Which island does the penguin with a bill length of 59.6 live on?",
    "How many penguins live on Dream island?",
    "What is the average body mass for each species?",
]
for q in questions:
    raw = text_to_sql(df, q)
    fixed = ground(df, raw)
    print(q)
    print("  model:   ", raw, "->", run_sql(df, raw))
    print("  grounded:", fixed, "->", run_sql(df, fixed), "\n")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

What is the heaviest body mass of a Gentoo penguin?
  model:    SELECT MAX("Body Mass (g)") FROM t WHERE "Species" = 'Gentoo penguin'; -> [[None]]
  grounded: SELECT MAX("Body Mass (g)") FROM t WHERE "Species" = 'Gentoo'; -> [[6300.0]] 

What is the average flipper length of Adelie penguins?
  model:    SELECT AVG("Flipper Length (mm)") FROM t WHERE "Species" = 'adelie penguins'; -> [[None]]
  grounded: SELECT AVG("Flipper Length (mm)") FROM t WHERE "Species" = 'Adelie'; -> [[189.95364238410596]] 

What is the lowest body mass on Torgersen island?
  model:    SELECT MIN("Body Mass (g)") FROM t WHERE "Species" = 'Torgersen'; -> [[None]]
  grounded: SELECT MIN("Body Mass (g)") FROM t WHERE "Island" = 'Torgersen'; -> [[2900.0]] 

Which island does the penguin with a bill length of 59.6 live on?
  model:    SELECT "Island" FROM t WHERE "Bill Length (mm)" = '59.6'; -> [['Biscoe']]
  grounded: SELECT "Island" FROM t WHERE "Bill Length (mm)" = '59.6'; -> [['Biscoe']] 

How many penguins live 